| Task ID | Task Name | Type | Category | Description |
|---------|-----------|------|----------|-------------|
| 2 | Transcription Factor Range | Binary | Gene Property Prediction | Differentiates between long-range and short-range transcription factors (TFs) |

In [ ]:
import os
import pandas as pd
import pickle
os.chdir(os.path.abspath(".."))
from utils.Gene_level_task_utils import evaluate_embeddings

# Read the regulatory range data
with open('Dataset/Gene_level_task/tf_regulatory_range.pickle', 'rb') as f:
    regulatory_data = pickle.load(f)


if os.path.exists("Dataset/Gene_level_task/tf_regulatory_range_data.pkl"):
    with open("Dataset/Gene_level_task/tf_regulatory_range_data.pkl", "rb") as f:
        converted_data = pickle.load(f)
    print("load tf_regulatory_range_data.pkl")
else:
    import mygene
    # Initialize the mygene client
    mg = mygene.MyGeneInfo()
    # Extract all Ensembl IDs from both lists
    lr_tfs = regulatory_data['long_range']
    sr_tfs = regulatory_data['short_range']

    # Combine both lists for batch query
    all_ids = lr_tfs + sr_tfs
    # Query mygene for gene info
    gene_info = mg.querymany(all_ids, scopes='ensembl.gene', fields=['symbol', 'name'], species='human')

    # Create a dictionary mapping Ensembl IDs to gene symbols
    id_to_symbol = {}
    for info in gene_info:
        if 'symbol' in info:
            id_to_symbol[info['query']] = info['symbol']

    # Convert lists to gene symbols
    lr_symbols = [id_to_symbol.get(ensembl_id, ensembl_id) for ensembl_id in lr_tfs]
    sr_symbols = [id_to_symbol.get(ensembl_id, ensembl_id) for ensembl_id in sr_tfs]

    # Create new dictionary with gene symbols
    converted_data = {
        'Long-range TFs': lr_symbols,
        'Short-range TFs': sr_symbols
    }
    with open("Dataset/Gene_level_task/tf_regulatory_range_data.pkl", "wb") as f:
        pickle.dump(converted_data, f)
    print("save tf_regulatory_range_data.pkl")

methods = ["SCGPT-HUMAN","SCGPT-PANCANCER","GF-6L30M","GF-20L95M","GF-12L95M","GF-12L30M", "GF-12L95MCANCER","GENE2VEC","TCGA-EMBEDDING","FROGS-ARCHS4","MUT2VEC","GENEPT-MODEL3","GENEPT-ADA","BIOCONCEPTVEC-SKIP-GRAM","BIOCONCEPTVEC-FASTTEXT","BIOCONCEPTVEC-GLOVE","BIOCONCEPTVEC-CBOW","CoxFormer"]
file_paths = {method: f'Embeddings/{method}.pkl' for method in methods}

# Check if the CSV file exists
csv_file_path = "Result/Gene_level_task/Transcription_Factor_Range.csv"
if os.path.exists(csv_file_path):
    # Load the existing results
    results_df = pd.read_csv(csv_file_path, index_col=0)

    # Check if all methods are already in the results
    existing_methods = results_df['Method'].tolist()
    additional_methods = [method for method in methods if method not in existing_methods]

    if additional_methods:
        print(f"Running evaluation for additional methods: {additional_methods}")

        # Evaluate the additional methods
        new_results_df = evaluate_embeddings(
            data_dict=converted_data,
            positive_label='Long-range TFs',
            negative_label='Short-range TFs',
            methods=additional_methods,
            file_paths=file_paths
        )

        # Append new results to the existing DataFrame
        results_df = pd.concat([results_df, new_results_df], ignore_index=True)
    else:
        print("All methods are already evaluated. Skipping evaluation.")
else:
    # If the CSV file does not exist, evaluate all methods
    print("CSV file does not exist. Running evaluation for all methods.")
    results_df = evaluate_embeddings(
        data_dict=converted_data,
        positive_label='Long-range TFs',
        negative_label='Short-range TFs',
        methods=methods,
        file_paths=file_paths
    )

# Save the updated results to CSV
results_df.to_csv(csv_file_path)

load tf_regulatory_range_data.pkl
All methods are already evaluated. Skipping evaluation.
